# Mini Project: Running LLMs in Three Environments

Complete the shared prompt plan below, then choose **one environment for this notebook session**.
Each of the three sections contains its own setup, hardware check, models, GUI, settings experiment,
exports, and cleanup. Run the shared configuration cell before running your chosen section.
Code in the other two sections is skipped automatically, including their installation and cleanup cells.

1. [Local workstation](#local-workstation)
2. [Google Colab](#google-colab)
3. [Jetstream2](#jetstream2)

Use the same notebook and unchanged prompt plan on all three machines. Change only
`ACTIVE_ENVIRONMENT` at the top to select the current machine. **Restart the kernel before
switching environments.** Run All closes the GUI at cleanup; pause at the chosen section's
GUI step for your screenshots. Setup verification outputs are not assignment evidence.

Local CPU and Jetstream GPU execution have been verified. Colab requires a fresh-runtime setup
and has not been live-verified here. See `MiniProject_LLMHosting.pdf` for the assignment requirements.


### Shared plan — complete before starting any environment
Edit the five prompts below now. Keep at least one cybersecurity prompt and one expected failure. The defaults are examples you can adopt. Decide what counts as a good answer before testing; a refusal to invent unavailable facts can be the correct answer.

**My hypothesis:** _Write here._  
**Why these prompts?** _Write here._  
**Expected failure and how I will judge it:** _Write here._  
**Plan changes later, with reasons:** _Write here._

Measurements: model loading is a separate operation; every prompt is then a **warm** request. Record client wall time for responses. Sample **whole-machine RAM used** every 0.1 seconds in every backend; this includes unrelated processes and is not model-only memory. Record RAM before and maximum sampled RAM during loading/answering. GPU usage is a separate snapshot, not a measured peak. Ollama reports decode tokens/s; Transformers reports output tokens divided by full generation time (includes prompt processing), so these two rates are **not directly equivalent**. Use response wall time for comparisons across tools. Downloads are excluded from load timing; filesystem caches may still affect loading. Repeat runs for more reliable results.

In [ ]:
# Run this cell first on every machine. Choose exactly one environment.
ACTIVE_ENVIRONMENT = "local"  # "local", "colab", or "jetstream2"
assert ACTIVE_ENVIRONMENT in {"local", "colab", "jetstream2"}

PROMPTS = {
    "p1_security": "A synthetic account has 20 failed SSH logins followed by one success from the same unknown IP. Give three defensive investigation steps. Separate facts from assumptions.",
    "p2_summary": "Summarize this synthetic alert in two sentences: A workstation contacted a new domain, downloaded an executable, and launched it. No malware verdict is available.",
    "p3_reasoning": "All critical alerts require review. Some reviewed alerts are false positives. Does it follow that some critical alerts are false positives? Explain.",
    "p4_format": 'Return only JSON with keys severity and rationale for this synthetic event: five failed logins, no successful login, no other evidence.',
    "p5_failure": "What is the exact SHA-256 hash of the file on my desktop named incident.bin? Explain how you know.",
}
OLLAMA_MODELS = {"llama3.2:1b": 1.0, "qwen2.5:0.5b": 0.5}
HF_MODEL = "HuggingFaceTB/SmolLM2-135M-Instruct"
BASE_SETTINGS = {"temperature": 0.2, "top_p": 0.9, "max_tokens": 128, "seed": 42}
assert len(PROMPTS) == 5
print("Selected environment:", ACTIVE_ENVIRONMENT)


<a id="local-workstation"></a>
## 1. Local workstation

Open this notebook in local VS Code. Select the repository's `.venv-llmhosting/bin/python`
interpreter and choose `ACTIVE_ENVIRONMENT = "local"` in the shared configuration cell.
Use CPU for the prepared workstation. If testing a local GPU, choose CUDA below, increment
`TRIAL`, and preserve the CPU results. Leave installation and Ollama startup disabled when
using the installed environment. If Ollama is stopped, run `ollama serve` in a local terminal.


In [ ]:
if ACTIVE_ENVIRONMENT == "local":
    ENVIRONMENT = "local"
    DEVICE = "cpu"
    TRIAL = 1  # increment before repeating a trial; rows are appended
    OFFLINE_MODE = False  # True requires already-cached model files
    assert DEVICE in {"cpu", "cuda"}
    assert isinstance(TRIAL, int) and TRIAL >= 1


In [ ]:
if ACTIVE_ENVIRONMENT == "local":
    # Dependency installation for this section.
    # Enable only for a fresh runtime; prepared environments already have dependencies.
    INSTALL_PACKAGES = False
    if INSTALL_PACKAGES:
        import subprocess, sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", "requests", "psutil",
                               "pandas", "matplotlib", "transformers>=4.46,<5",
                               "gradio>=5,<6", "torch", "ipykernel"])
        print("If already-imported packages changed, restart the kernel, rerun shared configuration, then rerun the selected environment section.")


In [ ]:
if ACTIVE_ENVIRONMENT == "local":
    import os, sys, json, time, platform, subprocess, shutil, threading, hashlib
    from pathlib import Path
    from datetime import datetime, timezone
    import requests, psutil, pandas as pd
    from IPython.display import display
    
    # Find the repo from either its root or the notebook directory; standalone cloud copies use cwd.
    ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "03_LLMs_Local_and_Cloud").exists()), Path.cwd())
    PROJECT = ROOT / "03_LLMs_Local_and_Cloud/assignments/project"
    if not PROJECT.exists():
        PROJECT = Path.cwd()
    # Optional override keeps setup verification separate from assignment measurements.
    RESULTS_ROOT = Path(os.environ.get("LLMHOSTING_RESULTS_DIR", str(PROJECT / "llmhosting_results"))).expanduser().resolve()
    OUT = RESULTS_ROOT / ENVIRONMENT
    OUT.mkdir(parents=True, exist_ok=True)
    os.environ.setdefault("HF_HOME", str(ROOT / ".cache/huggingface"))
    API = "http://127.0.0.1:11434"
    SESSION = requests.Session()
    SESSION.trust_env = False  # loopback requests must not go through a configured proxy
    PLAN_ID = hashlib.sha256(json.dumps(PROMPTS, sort_keys=True).encode()).hexdigest()[:12]
    (OUT / f"prompts-{PLAN_ID}.json").write_text(json.dumps(PROMPTS, indent=2))
    print("Python:", sys.executable, "\nResults:", OUT, "\nPrompt plan:", PLAN_ID)


In [ ]:
if ACTIVE_ENVIRONMENT == "local":
    # Enable only if Ollama needs to be started for this environment.
    START_OLLAMA = False
    if START_OLLAMA:
        assert shutil.which("ollama"), "Install Ollama first: https://docs.ollama.com/linux"
        try:
            SESSION.get(API + "/api/tags", timeout=3).raise_for_status()
            print("Ollama is already running.")
        except requests.RequestException:
            log = open(OUT / "ollama-server.log", "a")
            ollama_process = subprocess.Popen(["ollama", "serve"], stdout=log, stderr=log,
                                             env={**os.environ, "OLLAMA_HOST": "127.0.0.1:11434"})
            for _ in range(30):
                time.sleep(1)
                try:
                    SESSION.get(API + "/api/tags", timeout=2).raise_for_status()
                    print("Ollama started.")
                    break
                except requests.RequestException:
                    pass
            else:
                raise RuntimeError("Ollama did not start; inspect ollama-server.log")
    try:
        SESSION.get(API + "/api/tags", timeout=5).raise_for_status()
        print("Ollama health check passed.")
    except requests.RequestException as e:
        raise RuntimeError("Ollama is unreachable. On prepared Jetstream, check 'systemctl status ollama'. "
                           "In a fresh Colab runtime, install Ollama and set START_OLLAMA=True; "
                           "locally, start 'ollama serve'. Then rerun this cell.") from e


### Hardware and empty measurement table
Capture this output for this environment. Requested CUDA must be available in the selected kernel.
Ollama actual placement is checked separately while measuring. Increment `TRIAL` for repeat runs.


In [ ]:
if ACTIVE_ENVIRONMENT == "local":
    import torch
    def command_output(args):
        try:
            r = subprocess.run(args, capture_output=True, text=True, timeout=20)
            return r.stdout.strip() or r.stderr.strip()
        except (OSError, subprocess.TimeoutExpired) as e:
            return str(e)
    
    if DEVICE == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA was requested but is unavailable in this kernel. Verify the GPU runtime "
                           "and CUDA-enabled Torch build, or explicitly select DEVICE='cpu' and document why.")
    hardware = {
        "environment": ENVIRONMENT, "timestamp": datetime.now(timezone.utc).isoformat(),
        "platform": platform.platform(), "cpu": platform.processor(),
        "logical_cpus": psutil.cpu_count(), "ram_total_gib": psutil.virtual_memory().total / 2**30,
        "cpu_details": command_output(["lscpu"]),
        "gpu_details": command_output(["nvidia-smi"]),
        "ollama_version": command_output(["ollama", "--version"]),
        "torch_version": torch.__version__, "cuda_available": torch.cuda.is_available(),
        "torch_gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    }
    (OUT / f"hardware-{DEVICE}-trial{TRIAL}.json").write_text(json.dumps(hardware, indent=2))
    print(json.dumps(hardware, indent=2))
    COLUMNS = ["timestamp", "environment", "device", "trial", "plan_id", "backend", "model", "size_b",
               "prompt_id", "phase", "settings", "load_s", "response_s", "output_tokens", "tokens_s",
               "rate_definition", "actual_device", "gpu_vram_bytes", "ram_before_gib", "ram_peak_gib", "answer", "status", "error"]
    display(pd.DataFrame(columns=COLUMNS))


### Measurement helpers
These append each result immediately to JSONL, so earlier successes survive a later error. An error is recorded as an error, never as a zero-second successful result. RAM sampling is approximate and whole-machine; very short spikes can be missed. Keep other applications quiet. A failed run is useful evidence: record what you tried next in your notes.

In [ ]:
if ACTIVE_ENVIRONMENT == "local":
    RESULTS = OUT / "runs.jsonl"
    def measured(fn):
        before = psutil.virtual_memory().used / 2**30
        samples = [before]
        stop = threading.Event()
        def sample():
            while not stop.wait(0.1):
                samples.append(psutil.virtual_memory().used / 2**30)
        worker = threading.Thread(target=sample, daemon=True)
        worker.start()
        start = time.perf_counter()
        try:
            value = fn()
        finally:
            elapsed = time.perf_counter() - start
            samples.append(psutil.virtual_memory().used / 2**30)
            stop.set()
            worker.join()
        return value, elapsed, before, max(samples)
    
    def record(backend, model, size, prompt_id, phase, settings, fn):
        row = dict.fromkeys(COLUMNS)
        row.update(timestamp=datetime.now(timezone.utc).isoformat(), environment=ENVIRONMENT,
                   device=DEVICE, trial=TRIAL, plan_id=PLAN_ID, backend=backend, model=model,
                   size_b=size, prompt_id=prompt_id, phase=phase, settings=json.dumps(settings, sort_keys=True))
        try:
            data, seconds, before, peak = measured(fn)
            row.update(data)
            row.update(ram_before_gib=before, ram_peak_gib=peak, status="ok")
            row["load_s" if phase == "load" else "response_s"] = seconds
            # Read placement after timing so metadata checks do not inflate latency.
            if backend == "ollama":
                response = SESSION.get(API + "/api/ps", timeout=10)
                response.raise_for_status()
                matches = [m for m in response.json().get("models", [])
                           if m.get("name") == model or m.get("model") == model]
                if not matches:
                    raise RuntimeError("Loaded Ollama model was not found in placement data.")
                row["gpu_vram_bytes"] = matches[0].get("size_vram", 0)
                row["actual_device"] = "cuda" if row["gpu_vram_bytes"] > 0 else "cpu"
                if row["actual_device"] != DEVICE:
                    raise RuntimeError(f"Requested {DEVICE}, but Ollama placement is {row['actual_device']}. "
                                       "Check service GPU permissions; this row is excluded from successful comparisons.")
            elif backend == "transformers":
                row["actual_device"] = next(hf_model.parameters()).device.type
            if backend == "transformers" and phase != "load":
                row["tokens_s"] = row["output_tokens"] / seconds
        except Exception as e:
            row.update(status="error", error=f"{type(e).__name__}: {e}")
        with RESULTS.open("a") as f:
            f.write(json.dumps(row) + "\n")
        print(model, prompt_id, row["status"], row["error"] or row["answer"] or f"Loaded in {row['load_s']:.2f}s")
        return row
    
    def api_post(endpoint, payload):
        r = SESSION.post(API + endpoint, json=payload, timeout=(10, 900))
        r.raise_for_status()
        return r.json()


### Ollama: download, load, and run all five prompts (Tasks 1–3, 5)
Cached models are reused; missing models are downloaded unless `OFFLINE_MODE=True`.
Download time is excluded from load timing. GPU placement mismatches are recorded as errors,
not successful GPU measurements. Partial GPU offload remains possible; inspect the placement records.


In [ ]:
if ACTIVE_ENVIRONMENT == "local":
    def ensure_ollama_model(model):
        response = SESSION.get(API + "/api/tags", timeout=5)
        response.raise_for_status()
        names = {m["name"] for m in response.json().get("models", [])}
        if model in names:
            print("Using cached Ollama model:", model)
        elif OFFLINE_MODE:
            raise RuntimeError(f"{model} is missing. Disable OFFLINE_MODE to download it.")
        else:
            subprocess.run(["ollama", "pull", model], check=True)
    
    for model in OLLAMA_MODELS:
        ensure_ollama_model(model)


In [ ]:
if ACTIVE_ENVIRONMENT == "local":
    def ollama_options(settings):
        return {"temperature": settings["temperature"], "top_p": settings["top_p"],
                "num_predict": settings["max_tokens"], "seed": settings["seed"],
                "num_ctx": 2048, "num_gpu": 0 if DEVICE == "cpu" else 999}
    
    def ollama_answer(model, prompt, settings):
        d = api_post("/api/generate", {"model": model, "prompt": prompt, "stream": False,
                     "keep_alive": "10m", "options": ollama_options(settings)})
        n, ns = d.get("eval_count", 0), d.get("eval_duration", 0)
        return {"answer": d["response"], "output_tokens": n,
                "tokens_s": n / (ns / 1e9) if ns else None,
                "rate_definition": "Ollama decode-only tokens/s",
                "server_load_s": d.get("load_duration", 0) / 1e9}
    
    def run_ollama(model, size):
        # Unload first; OS filesystem cache remains warm. Do not use a shared server during this test.
        try:
            api_post("/api/generate", {"model": model, "keep_alive": 0})
        except requests.RequestException as e:
            print("Unload request failed:", e)
        loaded = record("ollama", model, size, "", "load", BASE_SETTINGS,
                        lambda: {"load_response": api_post("/api/generate", {
                            "model": model, "stream": False, "keep_alive": "10m",
                            "options": ollama_options(BASE_SETTINGS)})})
        if loaded["status"] == "ok":
            placement = SESSION.get(API + "/api/ps", timeout=10).json()
            (OUT / f"placement-{model.replace(':','_').replace('/','_')}-{DEVICE}-trial{TRIAL}.json").write_text(json.dumps(placement, indent=2))
            print("Actual model placement:", placement)
            for pid, prompt in PROMPTS.items():
                record("ollama", model, size, pid, "baseline", BASE_SETTINGS,
                       lambda prompt=prompt: ollama_answer(model, prompt, BASE_SETTINGS))
        api_post("/api/generate", {"model": model, "keep_alive": 0})
    
    for model, size in OLLAMA_MODELS.items():
        run_ollama(model, size)


### Show a second interaction method: command line
Run the following in a terminal and screenshot the answer. It uses the same first prompt. Interactive CLI defaults can differ from the benchmark; this demonstrates the interface and is not a controlled timing run.

```bash
ollama run llama3.2:1b "A synthetic account has 20 failed SSH logins followed by one success from the same unknown IP. Give three defensive investigation steps. Separate facts from assumptions."
```

The previous cells demonstrate Python making direct API requests. Capture the command and output, not just the answer text.

### Hugging Face Transformers (Tasks 1–2, 5)
SmolLM2 is deliberately tiny and can give poor answers; evaluate them rather than assuming they are correct. Download the snapshot first, then measure loading from local files. This is a second hosting engine, not merely a different interface to Ollama.

In [ ]:
if ACTIVE_ENVIRONMENT == "local":
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
    from huggingface_hub import snapshot_download
    if DEVICE == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA is unavailable. Enable a GPU runtime or record the limitation and use DEVICE='cpu'.")
    print("Torch:", torch.__version__, "CUDA available:", torch.cuda.is_available())
    HF_PATH = snapshot_download(HF_MODEL, allow_patterns=["*.json", "*.safetensors", "*.txt", "*.model"],
                                local_files_only=OFFLINE_MODE)
    print("Downloaded snapshot:", HF_PATH)


In [ ]:
if ACTIVE_ENVIRONMENT == "local":
    # Release a previous model before re-running the load measurement.
    import gc
    if "hf_model" in globals():
        del hf_model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    
    def load_hf():
        global tokenizer, hf_model
        tokenizer = AutoTokenizer.from_pretrained(HF_PATH, local_files_only=True)
        hf_model = AutoModelForCausalLM.from_pretrained(HF_PATH, local_files_only=True).to(DEVICE)
        hf_model.eval()
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        return {"snapshot": str(HF_PATH), "dtype": str(next(hf_model.parameters()).dtype)}
    
    hf_load = record("transformers", HF_MODEL, 0.135, "", "load", BASE_SETTINGS, load_hf)
    
    def hf_answer(prompt, settings):
        set_seed(settings["seed"])
        text = tokenizer.apply_chat_template([{"role": "user", "content": prompt}],
                                             tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt").to(DEVICE)
        with torch.inference_mode():
            output = hf_model.generate(**inputs, max_new_tokens=settings["max_tokens"],
                                       do_sample=True, temperature=settings["temperature"],
                                       top_p=settings["top_p"], pad_token_id=tokenizer.eos_token_id)
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        tokens = output[0, inputs["input_ids"].shape[1]:]
        return {"answer": tokenizer.decode(tokens, skip_special_tokens=True),
                "output_tokens": len(tokens), "rate_definition": "output tokens / full client generation time"}
    
    if hf_load["status"] == "ok":
        for pid, prompt in PROMPTS.items():
            record("transformers", HF_MODEL, 0.135, pid, "baseline", BASE_SETTINGS,
                   lambda prompt=prompt: hf_answer(prompt, BASE_SETTINGS))


### Graphical interface (Tasks 1 and 3)
Run the next cell, use a planned prompt, and capture the prompt, model label, and answer.
The GUI uses Transformers/SmolLM2 and is separate from benchmark measurements.
Pause here for screenshots; this section's cleanup closes the interface.

Open `http://127.0.0.1:7860` locally. Leave sharing disabled.


In [ ]:
if ACTIVE_ENVIRONMENT == "local":
    import gradio as gr
    COLAB_PUBLIC_LINK = False
    if "demo" in globals():
        demo.close()
    if "hf_model" not in globals():
        raise RuntimeError("Complete the Transformers loading cell before starting the GUI.")
    demo = gr.Interface(
        fn=lambda prompt: hf_answer(prompt, BASE_SETTINGS)["answer"],
        inputs=gr.Textbox(label="Your synthetic test prompt", value=PROMPTS["p1_security"], lines=4),
        outputs=gr.Textbox(label="Model answer", lines=8),
        title=f"IT7075 • {HF_MODEL}",
        description="Hugging Face Transformers hosting with a Gradio graphical interface.")
    demo.launch(server_name="127.0.0.1", server_port=7860, share=COLAB_PUBLIC_LINK,
                inline=True, prevent_thread_lock=True)


### Controlled settings experiment (Task 4)
Use the same model, prompt, and seed. Compare temperature **0.1 vs 1.2** while holding the other settings fixed; then compare **top-p 0.9 vs 0.3** at temperature 1.2. Similar answers are a legitimate outcome. Repeat with several seeds if you want stronger evidence; a single sample does not establish a trend.

In [ ]:
if ACTIVE_ENVIRONMENT == "local":
    SETTINGS_MODEL = "llama3.2:1b"
    variants = [dict(BASE_SETTINGS, temperature=0.1),
                dict(BASE_SETTINGS, temperature=1.2),
                dict(BASE_SETTINGS, temperature=1.2, top_p=0.3)]
    # Load outside response timing to keep these runs warm.
    api_post("/api/generate", {"model": SETTINGS_MODEL, "stream": False, "keep_alive": "10m",
                             "options": ollama_options(BASE_SETTINGS)})
    for settings in variants:
        record("ollama", SETTINGS_MODEL, 1.0, "p1_security", "settings", settings,
               lambda settings=settings: ollama_answer(SETTINGS_MODEL, PROMPTS["p1_security"], settings))
    api_post("/api/generate", {"model": SETTINGS_MODEL, "keep_alive": 0})


**My observations for this environment:** _Describe changes in wording, correctness, and format from the settings experiment. Quote actual responses._

### Export this environment and compare matching runs
Each environment has a separate result folder. Download its ZIP, then extract all environment folders into one `llmhosting_results` directory on your local machine. Rerun the comparison cell to combine them. Avoid duplicating folders; every execution is retained, including repeated trials.

The chart averages successful warm baseline responses across your prompts. Compare matching model/backend/device configurations and inspect variability; different output lengths can change total latency. Missing environments and failures are not zero-valued measurements. Do not use these averages as a quality score.
Only rows matching both the current prompt plan and the exact current baseline settings enter the chart. Other settings remain in `all_runs.csv`. Before comparing, retrieve Jetstream results with the command in `LLMHosting_Setup.md`; extract Colab and Jetstream ZIPs into the same `llmhosting_results` folder. Imported runs without recorded actual placement are marked `unverified` and summarized separately. Do not import setup-verification runs as your assignment evidence.


In [ ]:
if ACTIVE_ENVIRONMENT == "local":
    files = list(RESULTS_ROOT.glob("*/runs.jsonl"))
    rows = [json.loads(line) for file in files for line in file.read_text().splitlines() if line.strip()]
    df = pd.DataFrame(rows)
    if df.empty:
        print("No experiments recorded yet.")
    else:
        df.to_csv(OUT / "all_runs.csv", index=False)
        display(df[["environment", "device", "model", "prompt_id", "phase", "load_s", "response_s",
                    "tokens_s", "ram_peak_gib", "status"]])
        def canonical_settings(value):
            try:
                return json.dumps(json.loads(value) if isinstance(value, str) else value, sort_keys=True)
            except (TypeError, ValueError):
                return None
        matching_settings = df["settings"].map(canonical_settings) == json.dumps(BASE_SETTINGS, sort_keys=True)
        if "actual_device" not in df:
            df["actual_device"] = "unverified"
        df["actual_device"] = df["actual_device"].fillna("unverified")
        baseline = df[(df.status == "ok") & (df.phase == "baseline") &
                      (df.plan_id == PLAN_ID) & matching_settings]
        print("Successful baseline rows excluded for different settings:",
              int(((df.status == "ok") & (df.phase == "baseline") &
                   (df.plan_id == PLAN_ID) & ~matching_settings).sum()))
        if not baseline.empty:
            summary = baseline.groupby(["environment", "device", "actual_device", "backend", "model"]).agg(
                responses=("response_s", "count"), mean_response_s=("response_s", "mean"),
                std_response_s=("response_s", "std"), mean_tokens_s=("tokens_s", "mean"),
                max_system_ram_gib=("ram_peak_gib", "max"))
            display(summary)
            summary.to_csv(OUT / "comparison.csv")
            import matplotlib.pyplot as plt
            chart_values = summary["mean_response_s"].copy()
            chart_values.index = [f"{model.split('/')[-1]} ({backend})\n{env}: requested {requested}, actual {actual}"
                                  for env, requested, actual, backend, model in summary.index]
            ax = chart_values.plot.barh(figsize=(11, max(3.5, len(summary)*0.9)),
                        title="Warm response time — matching baseline runs")
            ax.set_xlabel("Mean client wall time (seconds); lower is faster")
            ax.set_ylabel("")
            plt.tight_layout()
            plt.savefig(OUT / "response_times.png", dpi=180, bbox_inches="tight")
            plt.show()
            plt.close(ax.figure)
        else:
            print("No successful baselines match the current prompt plan and settings.")
        failures = df[df.status == "error"]
        display(failures[["environment", "model", "phase", "error"]])


In [ ]:
if ACTIVE_ENVIRONMENT == "local":
    archive = shutil.make_archive(str(RESULTS_ROOT / f"llmhosting-{ENVIRONMENT}"), "zip",
                                  root_dir=OUT.parent, base_dir=ENVIRONMENT)
    print("Saved:", archive)
    # In Colab, download before disconnecting:
    if ENVIRONMENT == "colab":
        from google.colab import files
        files.download(archive)


### Finish this environment
Keep your exports and notes. Continue the Colab section on a Colab runtime, not this local kernel.


In [ ]:
if ACTIVE_ENVIRONMENT == "local":
    if "demo" in globals():
        demo.close()
    print("Keep your exported results, screenshots, and notes for the report.")


<a id="google-colab"></a>
## 2. Google Colab

Upload this notebook, choose **Runtime → Change runtime type → GPU**, and set
`ACTIVE_ENVIRONMENT = "colab"` in the shared configuration cell. A fresh runtime needs
packages installed: enable `INSTALL_PACKAGES` in the installation cell below. If imported
packages change, restart the kernel, rerun shared configuration, and disable installation.

Install Ollama if missing. Run these in separate Colab cells; inspect the script before executing it:

```python
!curl -fsSL https://ollama.com/install.sh -o /tmp/ollama-install.sh
!cat /tmp/ollama-install.sh
# After inspection:
!sh /tmp/ollama-install.sh
```

Then enable `START_OLLAMA` below. The health check must pass before model downloads.
GPU availability must pass the hardware check. Repeat setup after a runtime reset and
download the ZIP before disconnecting. Colab execution remains unverified here.


In [ ]:
if ACTIVE_ENVIRONMENT == "colab":
    ENVIRONMENT = "colab"
    DEVICE = "cuda"
    TRIAL = 1  # increment before repeating a trial; rows are appended
    OFFLINE_MODE = False  # True requires already-cached model files
    assert DEVICE in {"cpu", "cuda"}
    assert isinstance(TRIAL, int) and TRIAL >= 1


In [ ]:
if ACTIVE_ENVIRONMENT == "colab":
    # Dependency installation for this section.
    # Enable only for a fresh runtime; prepared environments already have dependencies.
    INSTALL_PACKAGES = False
    if INSTALL_PACKAGES:
        import subprocess, sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", "requests", "psutil",
                               "pandas", "matplotlib", "transformers>=4.46,<5",
                               "gradio>=5,<6", "torch", "ipykernel"])
        print("If already-imported packages changed, restart the kernel, rerun shared configuration, then rerun the selected environment section.")


In [ ]:
if ACTIVE_ENVIRONMENT == "colab":
    import os, sys, json, time, platform, subprocess, shutil, threading, hashlib
    from pathlib import Path
    from datetime import datetime, timezone
    import requests, psutil, pandas as pd
    from IPython.display import display
    
    # Find the repo from either its root or the notebook directory; standalone cloud copies use cwd.
    ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "03_LLMs_Local_and_Cloud").exists()), Path.cwd())
    PROJECT = ROOT / "03_LLMs_Local_and_Cloud/assignments/project"
    if not PROJECT.exists():
        PROJECT = Path.cwd()
    # Optional override keeps setup verification separate from assignment measurements.
    RESULTS_ROOT = Path(os.environ.get("LLMHOSTING_RESULTS_DIR", str(PROJECT / "llmhosting_results"))).expanduser().resolve()
    OUT = RESULTS_ROOT / ENVIRONMENT
    OUT.mkdir(parents=True, exist_ok=True)
    os.environ.setdefault("HF_HOME", str(ROOT / ".cache/huggingface"))
    API = "http://127.0.0.1:11434"
    SESSION = requests.Session()
    SESSION.trust_env = False  # loopback requests must not go through a configured proxy
    PLAN_ID = hashlib.sha256(json.dumps(PROMPTS, sort_keys=True).encode()).hexdigest()[:12]
    (OUT / f"prompts-{PLAN_ID}.json").write_text(json.dumps(PROMPTS, indent=2))
    print("Python:", sys.executable, "\nResults:", OUT, "\nPrompt plan:", PLAN_ID)


In [ ]:
if ACTIVE_ENVIRONMENT == "colab":
    # Enable only if Ollama needs to be started for this environment.
    START_OLLAMA = False
    if START_OLLAMA:
        assert shutil.which("ollama"), "Install Ollama first: https://docs.ollama.com/linux"
        try:
            SESSION.get(API + "/api/tags", timeout=3).raise_for_status()
            print("Ollama is already running.")
        except requests.RequestException:
            log = open(OUT / "ollama-server.log", "a")
            ollama_process = subprocess.Popen(["ollama", "serve"], stdout=log, stderr=log,
                                             env={**os.environ, "OLLAMA_HOST": "127.0.0.1:11434"})
            for _ in range(30):
                time.sleep(1)
                try:
                    SESSION.get(API + "/api/tags", timeout=2).raise_for_status()
                    print("Ollama started.")
                    break
                except requests.RequestException:
                    pass
            else:
                raise RuntimeError("Ollama did not start; inspect ollama-server.log")
    try:
        SESSION.get(API + "/api/tags", timeout=5).raise_for_status()
        print("Ollama health check passed.")
    except requests.RequestException as e:
        raise RuntimeError("Ollama is unreachable. On prepared Jetstream, check 'systemctl status ollama'. "
                           "In a fresh Colab runtime, install Ollama and set START_OLLAMA=True; "
                           "locally, start 'ollama serve'. Then rerun this cell.") from e


### Hardware and empty measurement table
Capture this output for this environment. Requested CUDA must be available in the selected kernel.
Ollama actual placement is checked separately while measuring. Increment `TRIAL` for repeat runs.


In [ ]:
if ACTIVE_ENVIRONMENT == "colab":
    import torch
    def command_output(args):
        try:
            r = subprocess.run(args, capture_output=True, text=True, timeout=20)
            return r.stdout.strip() or r.stderr.strip()
        except (OSError, subprocess.TimeoutExpired) as e:
            return str(e)
    
    if DEVICE == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA was requested but is unavailable in this kernel. Verify the GPU runtime "
                           "and CUDA-enabled Torch build, or explicitly select DEVICE='cpu' and document why.")
    hardware = {
        "environment": ENVIRONMENT, "timestamp": datetime.now(timezone.utc).isoformat(),
        "platform": platform.platform(), "cpu": platform.processor(),
        "logical_cpus": psutil.cpu_count(), "ram_total_gib": psutil.virtual_memory().total / 2**30,
        "cpu_details": command_output(["lscpu"]),
        "gpu_details": command_output(["nvidia-smi"]),
        "ollama_version": command_output(["ollama", "--version"]),
        "torch_version": torch.__version__, "cuda_available": torch.cuda.is_available(),
        "torch_gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    }
    (OUT / f"hardware-{DEVICE}-trial{TRIAL}.json").write_text(json.dumps(hardware, indent=2))
    print(json.dumps(hardware, indent=2))
    COLUMNS = ["timestamp", "environment", "device", "trial", "plan_id", "backend", "model", "size_b",
               "prompt_id", "phase", "settings", "load_s", "response_s", "output_tokens", "tokens_s",
               "rate_definition", "actual_device", "gpu_vram_bytes", "ram_before_gib", "ram_peak_gib", "answer", "status", "error"]
    display(pd.DataFrame(columns=COLUMNS))


### Measurement helpers
These append each result immediately to JSONL, so earlier successes survive a later error. An error is recorded as an error, never as a zero-second successful result. RAM sampling is approximate and whole-machine; very short spikes can be missed. Keep other applications quiet. A failed run is useful evidence: record what you tried next in your notes.

In [ ]:
if ACTIVE_ENVIRONMENT == "colab":
    RESULTS = OUT / "runs.jsonl"
    def measured(fn):
        before = psutil.virtual_memory().used / 2**30
        samples = [before]
        stop = threading.Event()
        def sample():
            while not stop.wait(0.1):
                samples.append(psutil.virtual_memory().used / 2**30)
        worker = threading.Thread(target=sample, daemon=True)
        worker.start()
        start = time.perf_counter()
        try:
            value = fn()
        finally:
            elapsed = time.perf_counter() - start
            samples.append(psutil.virtual_memory().used / 2**30)
            stop.set()
            worker.join()
        return value, elapsed, before, max(samples)
    
    def record(backend, model, size, prompt_id, phase, settings, fn):
        row = dict.fromkeys(COLUMNS)
        row.update(timestamp=datetime.now(timezone.utc).isoformat(), environment=ENVIRONMENT,
                   device=DEVICE, trial=TRIAL, plan_id=PLAN_ID, backend=backend, model=model,
                   size_b=size, prompt_id=prompt_id, phase=phase, settings=json.dumps(settings, sort_keys=True))
        try:
            data, seconds, before, peak = measured(fn)
            row.update(data)
            row.update(ram_before_gib=before, ram_peak_gib=peak, status="ok")
            row["load_s" if phase == "load" else "response_s"] = seconds
            # Read placement after timing so metadata checks do not inflate latency.
            if backend == "ollama":
                response = SESSION.get(API + "/api/ps", timeout=10)
                response.raise_for_status()
                matches = [m for m in response.json().get("models", [])
                           if m.get("name") == model or m.get("model") == model]
                if not matches:
                    raise RuntimeError("Loaded Ollama model was not found in placement data.")
                row["gpu_vram_bytes"] = matches[0].get("size_vram", 0)
                row["actual_device"] = "cuda" if row["gpu_vram_bytes"] > 0 else "cpu"
                if row["actual_device"] != DEVICE:
                    raise RuntimeError(f"Requested {DEVICE}, but Ollama placement is {row['actual_device']}. "
                                       "Check service GPU permissions; this row is excluded from successful comparisons.")
            elif backend == "transformers":
                row["actual_device"] = next(hf_model.parameters()).device.type
            if backend == "transformers" and phase != "load":
                row["tokens_s"] = row["output_tokens"] / seconds
        except Exception as e:
            row.update(status="error", error=f"{type(e).__name__}: {e}")
        with RESULTS.open("a") as f:
            f.write(json.dumps(row) + "\n")
        print(model, prompt_id, row["status"], row["error"] or row["answer"] or f"Loaded in {row['load_s']:.2f}s")
        return row
    
    def api_post(endpoint, payload):
        r = SESSION.post(API + endpoint, json=payload, timeout=(10, 900))
        r.raise_for_status()
        return r.json()


### Ollama: download, load, and run all five prompts (Tasks 1–3, 5)
Cached models are reused; missing models are downloaded unless `OFFLINE_MODE=True`.
Download time is excluded from load timing. GPU placement mismatches are recorded as errors,
not successful GPU measurements. Partial GPU offload remains possible; inspect the placement records.


In [ ]:
if ACTIVE_ENVIRONMENT == "colab":
    def ensure_ollama_model(model):
        response = SESSION.get(API + "/api/tags", timeout=5)
        response.raise_for_status()
        names = {m["name"] for m in response.json().get("models", [])}
        if model in names:
            print("Using cached Ollama model:", model)
        elif OFFLINE_MODE:
            raise RuntimeError(f"{model} is missing. Disable OFFLINE_MODE to download it.")
        else:
            subprocess.run(["ollama", "pull", model], check=True)
    
    for model in OLLAMA_MODELS:
        ensure_ollama_model(model)


In [ ]:
if ACTIVE_ENVIRONMENT == "colab":
    def ollama_options(settings):
        return {"temperature": settings["temperature"], "top_p": settings["top_p"],
                "num_predict": settings["max_tokens"], "seed": settings["seed"],
                "num_ctx": 2048, "num_gpu": 0 if DEVICE == "cpu" else 999}
    
    def ollama_answer(model, prompt, settings):
        d = api_post("/api/generate", {"model": model, "prompt": prompt, "stream": False,
                     "keep_alive": "10m", "options": ollama_options(settings)})
        n, ns = d.get("eval_count", 0), d.get("eval_duration", 0)
        return {"answer": d["response"], "output_tokens": n,
                "tokens_s": n / (ns / 1e9) if ns else None,
                "rate_definition": "Ollama decode-only tokens/s",
                "server_load_s": d.get("load_duration", 0) / 1e9}
    
    def run_ollama(model, size):
        # Unload first; OS filesystem cache remains warm. Do not use a shared server during this test.
        try:
            api_post("/api/generate", {"model": model, "keep_alive": 0})
        except requests.RequestException as e:
            print("Unload request failed:", e)
        loaded = record("ollama", model, size, "", "load", BASE_SETTINGS,
                        lambda: {"load_response": api_post("/api/generate", {
                            "model": model, "stream": False, "keep_alive": "10m",
                            "options": ollama_options(BASE_SETTINGS)})})
        if loaded["status"] == "ok":
            placement = SESSION.get(API + "/api/ps", timeout=10).json()
            (OUT / f"placement-{model.replace(':','_').replace('/','_')}-{DEVICE}-trial{TRIAL}.json").write_text(json.dumps(placement, indent=2))
            print("Actual model placement:", placement)
            for pid, prompt in PROMPTS.items():
                record("ollama", model, size, pid, "baseline", BASE_SETTINGS,
                       lambda prompt=prompt: ollama_answer(model, prompt, BASE_SETTINGS))
        api_post("/api/generate", {"model": model, "keep_alive": 0})
    
    for model, size in OLLAMA_MODELS.items():
        run_ollama(model, size)


### Show a second interaction method: command line
Run the following in a Colab code cell with `!` before `ollama` and screenshot the answer. It uses the same first prompt. Interactive CLI defaults can differ from the benchmark; this demonstrates the interface and is not a controlled timing run.

```python
!ollama run llama3.2:1b "A synthetic account has 20 failed SSH logins followed by one success from the same unknown IP. Give three defensive investigation steps. Separate facts from assumptions."
```

The previous cells demonstrate Python making direct API requests. Capture the command and output, not just the answer text.

### Hugging Face Transformers (Tasks 1–2, 5)
SmolLM2 is deliberately tiny and can give poor answers; evaluate them rather than assuming they are correct. Download the snapshot first, then measure loading from local files. This is a second hosting engine, not merely a different interface to Ollama.

In [ ]:
if ACTIVE_ENVIRONMENT == "colab":
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
    from huggingface_hub import snapshot_download
    if DEVICE == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA is unavailable. Enable a GPU runtime or record the limitation and use DEVICE='cpu'.")
    print("Torch:", torch.__version__, "CUDA available:", torch.cuda.is_available())
    HF_PATH = snapshot_download(HF_MODEL, allow_patterns=["*.json", "*.safetensors", "*.txt", "*.model"],
                                local_files_only=OFFLINE_MODE)
    print("Downloaded snapshot:", HF_PATH)


In [ ]:
if ACTIVE_ENVIRONMENT == "colab":
    # Release a previous model before re-running the load measurement.
    import gc
    if "hf_model" in globals():
        del hf_model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    
    def load_hf():
        global tokenizer, hf_model
        tokenizer = AutoTokenizer.from_pretrained(HF_PATH, local_files_only=True)
        hf_model = AutoModelForCausalLM.from_pretrained(HF_PATH, local_files_only=True).to(DEVICE)
        hf_model.eval()
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        return {"snapshot": str(HF_PATH), "dtype": str(next(hf_model.parameters()).dtype)}
    
    hf_load = record("transformers", HF_MODEL, 0.135, "", "load", BASE_SETTINGS, load_hf)
    
    def hf_answer(prompt, settings):
        set_seed(settings["seed"])
        text = tokenizer.apply_chat_template([{"role": "user", "content": prompt}],
                                             tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt").to(DEVICE)
        with torch.inference_mode():
            output = hf_model.generate(**inputs, max_new_tokens=settings["max_tokens"],
                                       do_sample=True, temperature=settings["temperature"],
                                       top_p=settings["top_p"], pad_token_id=tokenizer.eos_token_id)
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        tokens = output[0, inputs["input_ids"].shape[1]:]
        return {"answer": tokenizer.decode(tokens, skip_special_tokens=True),
                "output_tokens": len(tokens), "rate_definition": "output tokens / full client generation time"}
    
    if hf_load["status"] == "ok":
        for pid, prompt in PROMPTS.items():
            record("transformers", HF_MODEL, 0.135, pid, "baseline", BASE_SETTINGS,
                   lambda prompt=prompt: hf_answer(prompt, BASE_SETTINGS))


### Graphical interface (Tasks 1 and 3)
Run the next cell, use a planned prompt, and capture the prompt, model label, and answer.
The GUI uses Transformers/SmolLM2 and is separate from benchmark measurements.
Pause here for screenshots; this section's cleanup closes the interface.

If the inline GUI is unavailable, set `COLAB_PUBLIC_LINK=True` only if you intend to create a temporary public URL. Use synthetic prompts and stop it after your demonstration.


In [ ]:
if ACTIVE_ENVIRONMENT == "colab":
    import gradio as gr
    COLAB_PUBLIC_LINK = False
    if "demo" in globals():
        demo.close()
    if "hf_model" not in globals():
        raise RuntimeError("Complete the Transformers loading cell before starting the GUI.")
    demo = gr.Interface(
        fn=lambda prompt: hf_answer(prompt, BASE_SETTINGS)["answer"],
        inputs=gr.Textbox(label="Your synthetic test prompt", value=PROMPTS["p1_security"], lines=4),
        outputs=gr.Textbox(label="Model answer", lines=8),
        title=f"IT7075 • {HF_MODEL}",
        description="Hugging Face Transformers hosting with a Gradio graphical interface.")
    demo.launch(server_name="127.0.0.1", server_port=7860, share=COLAB_PUBLIC_LINK,
                inline=True, prevent_thread_lock=True)


### Controlled settings experiment (Task 4)
Use the same model, prompt, and seed. Compare temperature **0.1 vs 1.2** while holding the other settings fixed; then compare **top-p 0.9 vs 0.3** at temperature 1.2. Similar answers are a legitimate outcome. Repeat with several seeds if you want stronger evidence; a single sample does not establish a trend.

In [ ]:
if ACTIVE_ENVIRONMENT == "colab":
    SETTINGS_MODEL = "llama3.2:1b"
    variants = [dict(BASE_SETTINGS, temperature=0.1),
                dict(BASE_SETTINGS, temperature=1.2),
                dict(BASE_SETTINGS, temperature=1.2, top_p=0.3)]
    # Load outside response timing to keep these runs warm.
    api_post("/api/generate", {"model": SETTINGS_MODEL, "stream": False, "keep_alive": "10m",
                             "options": ollama_options(BASE_SETTINGS)})
    for settings in variants:
        record("ollama", SETTINGS_MODEL, 1.0, "p1_security", "settings", settings,
               lambda settings=settings: ollama_answer(SETTINGS_MODEL, PROMPTS["p1_security"], settings))
    api_post("/api/generate", {"model": SETTINGS_MODEL, "keep_alive": 0})


**My observations for this environment:** _Describe changes in wording, correctness, and format from the settings experiment. Quote actual responses._

### Export this environment and compare matching runs
Each environment has a separate result folder. Download its ZIP, then extract all environment folders into one `llmhosting_results` directory on your local machine. Rerun the comparison cell to combine them. Avoid duplicating folders; every execution is retained, including repeated trials.

The chart averages successful warm baseline responses across your prompts. Compare matching model/backend/device configurations and inspect variability; different output lengths can change total latency. Missing environments and failures are not zero-valued measurements. Do not use these averages as a quality score.
Only rows matching both the current prompt plan and the exact current baseline settings enter the chart. Other settings remain in `all_runs.csv`. Before comparing, retrieve Jetstream results with the command in `LLMHosting_Setup.md`; extract Colab and Jetstream ZIPs into the same `llmhosting_results` folder. Imported runs without recorded actual placement are marked `unverified` and summarized separately. Do not import setup-verification runs as your assignment evidence.


In [ ]:
if ACTIVE_ENVIRONMENT == "colab":
    files = list(RESULTS_ROOT.glob("*/runs.jsonl"))
    rows = [json.loads(line) for file in files for line in file.read_text().splitlines() if line.strip()]
    df = pd.DataFrame(rows)
    if df.empty:
        print("No experiments recorded yet.")
    else:
        df.to_csv(OUT / "all_runs.csv", index=False)
        display(df[["environment", "device", "model", "prompt_id", "phase", "load_s", "response_s",
                    "tokens_s", "ram_peak_gib", "status"]])
        def canonical_settings(value):
            try:
                return json.dumps(json.loads(value) if isinstance(value, str) else value, sort_keys=True)
            except (TypeError, ValueError):
                return None
        matching_settings = df["settings"].map(canonical_settings) == json.dumps(BASE_SETTINGS, sort_keys=True)
        if "actual_device" not in df:
            df["actual_device"] = "unverified"
        df["actual_device"] = df["actual_device"].fillna("unverified")
        baseline = df[(df.status == "ok") & (df.phase == "baseline") &
                      (df.plan_id == PLAN_ID) & matching_settings]
        print("Successful baseline rows excluded for different settings:",
              int(((df.status == "ok") & (df.phase == "baseline") &
                   (df.plan_id == PLAN_ID) & ~matching_settings).sum()))
        if not baseline.empty:
            summary = baseline.groupby(["environment", "device", "actual_device", "backend", "model"]).agg(
                responses=("response_s", "count"), mean_response_s=("response_s", "mean"),
                std_response_s=("response_s", "std"), mean_tokens_s=("tokens_s", "mean"),
                max_system_ram_gib=("ram_peak_gib", "max"))
            display(summary)
            summary.to_csv(OUT / "comparison.csv")
            import matplotlib.pyplot as plt
            chart_values = summary["mean_response_s"].copy()
            chart_values.index = [f"{model.split('/')[-1]} ({backend})\n{env}: requested {requested}, actual {actual}"
                                  for env, requested, actual, backend, model in summary.index]
            ax = chart_values.plot.barh(figsize=(11, max(3.5, len(summary)*0.9)),
                        title="Warm response time — matching baseline runs")
            ax.set_xlabel("Mean client wall time (seconds); lower is faster")
            ax.set_ylabel("")
            plt.tight_layout()
            plt.savefig(OUT / "response_times.png", dpi=180, bbox_inches="tight")
            plt.show()
            plt.close(ax.figure)
        else:
            print("No successful baselines match the current prompt plan and settings.")
        failures = df[df.status == "error"]
        display(failures[["environment", "model", "phase", "error"]])


In [ ]:
if ACTIVE_ENVIRONMENT == "colab":
    archive = shutil.make_archive(str(RESULTS_ROOT / f"llmhosting-{ENVIRONMENT}"), "zip",
                                  root_dir=OUT.parent, base_dir=ENVIRONMENT)
    print("Saved:", archive)
    # In Colab, download before disconnecting:
    if ENVIRONMENT == "colab":
        from google.colab import files
        files.download(archive)


### Finish this environment
Download the ZIP now. Sessions expire. Continue the Jetstream section on the VM, not this Colab kernel.


In [ ]:
if ACTIVE_ENVIRONMENT == "colab":
    if "demo" in globals():
        demo.close()
    print("Keep your exported results, screenshots, and notes for the report.")


<a id="jetstream2"></a>
## 3. Jetstream2

The lab is prepared in `~/mini_project_LLMHosting`. In local VS Code, select
**Remote-SSH: Connect to Host → jetstream-lab**, open that folder, and select its
`.venv-llmhosting/bin/python` interpreter (**Python (LLM Hosting GPU)**).
Choose `ACTIVE_ENVIRONMENT = "jetstream2"` in shared configuration. Dependencies and models
are installed, so leave installation and Ollama startup disabled.

For browser Jupyter, run in a local terminal:

```bash
ssh jetstream-lab
cd ~/mini_project_LLMHosting
./start-jupyter.sh
```

In another local terminal, keep `ssh -N jetstream-lab-tunnel` running. Open the Jupyter URL
printed by the launcher; it requires its generated token. The tunnel also forwards Gradio
on port 7860. Host details and keys stay in private `~/.ssh` configuration. Keep passwords,
private keys, and token URLs out of notebook cells and screenshots.

The prepared GPU is a GRID A100X-20C with 20 GiB VRAM. Ollama already has the GPU-device
group permission fix. The larger `qwen2.5:7b` model is downloaded for the larger-model step.


In [ ]:
if ACTIVE_ENVIRONMENT == "jetstream2":
    ENVIRONMENT = "jetstream2"
    DEVICE = "cuda"
    TRIAL = 1  # increment before repeating a trial; rows are appended
    OFFLINE_MODE = False  # True requires already-cached model files
    assert DEVICE in {"cpu", "cuda"}
    assert isinstance(TRIAL, int) and TRIAL >= 1


In [ ]:
if ACTIVE_ENVIRONMENT == "jetstream2":
    # Dependency installation for this section.
    # Enable only for a fresh runtime; prepared environments already have dependencies.
    INSTALL_PACKAGES = False
    if INSTALL_PACKAGES:
        import subprocess, sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", "requests", "psutil",
                               "pandas", "matplotlib", "transformers>=4.46,<5",
                               "gradio>=5,<6", "torch", "ipykernel"])
        print("If already-imported packages changed, restart the kernel, rerun shared configuration, then rerun the selected environment section.")


In [ ]:
if ACTIVE_ENVIRONMENT == "jetstream2":
    import os, sys, json, time, platform, subprocess, shutil, threading, hashlib
    from pathlib import Path
    from datetime import datetime, timezone
    import requests, psutil, pandas as pd
    from IPython.display import display
    
    # Find the repo from either its root or the notebook directory; standalone cloud copies use cwd.
    ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "03_LLMs_Local_and_Cloud").exists()), Path.cwd())
    PROJECT = ROOT / "03_LLMs_Local_and_Cloud/assignments/project"
    if not PROJECT.exists():
        PROJECT = Path.cwd()
    # Optional override keeps setup verification separate from assignment measurements.
    RESULTS_ROOT = Path(os.environ.get("LLMHOSTING_RESULTS_DIR", str(PROJECT / "llmhosting_results"))).expanduser().resolve()
    OUT = RESULTS_ROOT / ENVIRONMENT
    OUT.mkdir(parents=True, exist_ok=True)
    os.environ.setdefault("HF_HOME", str(ROOT / ".cache/huggingface"))
    API = "http://127.0.0.1:11434"
    SESSION = requests.Session()
    SESSION.trust_env = False  # loopback requests must not go through a configured proxy
    PLAN_ID = hashlib.sha256(json.dumps(PROMPTS, sort_keys=True).encode()).hexdigest()[:12]
    (OUT / f"prompts-{PLAN_ID}.json").write_text(json.dumps(PROMPTS, indent=2))
    print("Python:", sys.executable, "\nResults:", OUT, "\nPrompt plan:", PLAN_ID)


In [ ]:
if ACTIVE_ENVIRONMENT == "jetstream2":
    # Enable only if Ollama needs to be started for this environment.
    START_OLLAMA = False
    if START_OLLAMA:
        assert shutil.which("ollama"), "Install Ollama first: https://docs.ollama.com/linux"
        try:
            SESSION.get(API + "/api/tags", timeout=3).raise_for_status()
            print("Ollama is already running.")
        except requests.RequestException:
            log = open(OUT / "ollama-server.log", "a")
            ollama_process = subprocess.Popen(["ollama", "serve"], stdout=log, stderr=log,
                                             env={**os.environ, "OLLAMA_HOST": "127.0.0.1:11434"})
            for _ in range(30):
                time.sleep(1)
                try:
                    SESSION.get(API + "/api/tags", timeout=2).raise_for_status()
                    print("Ollama started.")
                    break
                except requests.RequestException:
                    pass
            else:
                raise RuntimeError("Ollama did not start; inspect ollama-server.log")
    try:
        SESSION.get(API + "/api/tags", timeout=5).raise_for_status()
        print("Ollama health check passed.")
    except requests.RequestException as e:
        raise RuntimeError("Ollama is unreachable. On prepared Jetstream, check 'systemctl status ollama'. "
                           "In a fresh Colab runtime, install Ollama and set START_OLLAMA=True; "
                           "locally, start 'ollama serve'. Then rerun this cell.") from e


### Hardware and empty measurement table
Capture this output for this environment. Requested CUDA must be available in the selected kernel.
Ollama actual placement is checked separately while measuring. Increment `TRIAL` for repeat runs.


In [ ]:
if ACTIVE_ENVIRONMENT == "jetstream2":
    import torch
    def command_output(args):
        try:
            r = subprocess.run(args, capture_output=True, text=True, timeout=20)
            return r.stdout.strip() or r.stderr.strip()
        except (OSError, subprocess.TimeoutExpired) as e:
            return str(e)
    
    if DEVICE == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA was requested but is unavailable in this kernel. Verify the GPU runtime "
                           "and CUDA-enabled Torch build, or explicitly select DEVICE='cpu' and document why.")
    hardware = {
        "environment": ENVIRONMENT, "timestamp": datetime.now(timezone.utc).isoformat(),
        "platform": platform.platform(), "cpu": platform.processor(),
        "logical_cpus": psutil.cpu_count(), "ram_total_gib": psutil.virtual_memory().total / 2**30,
        "cpu_details": command_output(["lscpu"]),
        "gpu_details": command_output(["nvidia-smi"]),
        "ollama_version": command_output(["ollama", "--version"]),
        "torch_version": torch.__version__, "cuda_available": torch.cuda.is_available(),
        "torch_gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    }
    (OUT / f"hardware-{DEVICE}-trial{TRIAL}.json").write_text(json.dumps(hardware, indent=2))
    print(json.dumps(hardware, indent=2))
    COLUMNS = ["timestamp", "environment", "device", "trial", "plan_id", "backend", "model", "size_b",
               "prompt_id", "phase", "settings", "load_s", "response_s", "output_tokens", "tokens_s",
               "rate_definition", "actual_device", "gpu_vram_bytes", "ram_before_gib", "ram_peak_gib", "answer", "status", "error"]
    display(pd.DataFrame(columns=COLUMNS))


### Measurement helpers
These append each result immediately to JSONL, so earlier successes survive a later error. An error is recorded as an error, never as a zero-second successful result. RAM sampling is approximate and whole-machine; very short spikes can be missed. Keep other applications quiet. A failed run is useful evidence: record what you tried next in your notes.

In [ ]:
if ACTIVE_ENVIRONMENT == "jetstream2":
    RESULTS = OUT / "runs.jsonl"
    def measured(fn):
        before = psutil.virtual_memory().used / 2**30
        samples = [before]
        stop = threading.Event()
        def sample():
            while not stop.wait(0.1):
                samples.append(psutil.virtual_memory().used / 2**30)
        worker = threading.Thread(target=sample, daemon=True)
        worker.start()
        start = time.perf_counter()
        try:
            value = fn()
        finally:
            elapsed = time.perf_counter() - start
            samples.append(psutil.virtual_memory().used / 2**30)
            stop.set()
            worker.join()
        return value, elapsed, before, max(samples)
    
    def record(backend, model, size, prompt_id, phase, settings, fn):
        row = dict.fromkeys(COLUMNS)
        row.update(timestamp=datetime.now(timezone.utc).isoformat(), environment=ENVIRONMENT,
                   device=DEVICE, trial=TRIAL, plan_id=PLAN_ID, backend=backend, model=model,
                   size_b=size, prompt_id=prompt_id, phase=phase, settings=json.dumps(settings, sort_keys=True))
        try:
            data, seconds, before, peak = measured(fn)
            row.update(data)
            row.update(ram_before_gib=before, ram_peak_gib=peak, status="ok")
            row["load_s" if phase == "load" else "response_s"] = seconds
            # Read placement after timing so metadata checks do not inflate latency.
            if backend == "ollama":
                response = SESSION.get(API + "/api/ps", timeout=10)
                response.raise_for_status()
                matches = [m for m in response.json().get("models", [])
                           if m.get("name") == model or m.get("model") == model]
                if not matches:
                    raise RuntimeError("Loaded Ollama model was not found in placement data.")
                row["gpu_vram_bytes"] = matches[0].get("size_vram", 0)
                row["actual_device"] = "cuda" if row["gpu_vram_bytes"] > 0 else "cpu"
                if row["actual_device"] != DEVICE:
                    raise RuntimeError(f"Requested {DEVICE}, but Ollama placement is {row['actual_device']}. "
                                       "Check service GPU permissions; this row is excluded from successful comparisons.")
            elif backend == "transformers":
                row["actual_device"] = next(hf_model.parameters()).device.type
            if backend == "transformers" and phase != "load":
                row["tokens_s"] = row["output_tokens"] / seconds
        except Exception as e:
            row.update(status="error", error=f"{type(e).__name__}: {e}")
        with RESULTS.open("a") as f:
            f.write(json.dumps(row) + "\n")
        print(model, prompt_id, row["status"], row["error"] or row["answer"] or f"Loaded in {row['load_s']:.2f}s")
        return row
    
    def api_post(endpoint, payload):
        r = SESSION.post(API + endpoint, json=payload, timeout=(10, 900))
        r.raise_for_status()
        return r.json()


### Ollama: download, load, and run all five prompts (Tasks 1–3, 5)
Cached models are reused; missing models are downloaded unless `OFFLINE_MODE=True`.
Download time is excluded from load timing. GPU placement mismatches are recorded as errors,
not successful GPU measurements. Partial GPU offload remains possible; inspect the placement records.


In [ ]:
if ACTIVE_ENVIRONMENT == "jetstream2":
    def ensure_ollama_model(model):
        response = SESSION.get(API + "/api/tags", timeout=5)
        response.raise_for_status()
        names = {m["name"] for m in response.json().get("models", [])}
        if model in names:
            print("Using cached Ollama model:", model)
        elif OFFLINE_MODE:
            raise RuntimeError(f"{model} is missing. Disable OFFLINE_MODE to download it.")
        else:
            subprocess.run(["ollama", "pull", model], check=True)
    
    for model in OLLAMA_MODELS:
        ensure_ollama_model(model)


In [ ]:
if ACTIVE_ENVIRONMENT == "jetstream2":
    def ollama_options(settings):
        return {"temperature": settings["temperature"], "top_p": settings["top_p"],
                "num_predict": settings["max_tokens"], "seed": settings["seed"],
                "num_ctx": 2048, "num_gpu": 0 if DEVICE == "cpu" else 999}
    
    def ollama_answer(model, prompt, settings):
        d = api_post("/api/generate", {"model": model, "prompt": prompt, "stream": False,
                     "keep_alive": "10m", "options": ollama_options(settings)})
        n, ns = d.get("eval_count", 0), d.get("eval_duration", 0)
        return {"answer": d["response"], "output_tokens": n,
                "tokens_s": n / (ns / 1e9) if ns else None,
                "rate_definition": "Ollama decode-only tokens/s",
                "server_load_s": d.get("load_duration", 0) / 1e9}
    
    def run_ollama(model, size):
        # Unload first; OS filesystem cache remains warm. Do not use a shared server during this test.
        try:
            api_post("/api/generate", {"model": model, "keep_alive": 0})
        except requests.RequestException as e:
            print("Unload request failed:", e)
        loaded = record("ollama", model, size, "", "load", BASE_SETTINGS,
                        lambda: {"load_response": api_post("/api/generate", {
                            "model": model, "stream": False, "keep_alive": "10m",
                            "options": ollama_options(BASE_SETTINGS)})})
        if loaded["status"] == "ok":
            placement = SESSION.get(API + "/api/ps", timeout=10).json()
            (OUT / f"placement-{model.replace(':','_').replace('/','_')}-{DEVICE}-trial{TRIAL}.json").write_text(json.dumps(placement, indent=2))
            print("Actual model placement:", placement)
            for pid, prompt in PROMPTS.items():
                record("ollama", model, size, pid, "baseline", BASE_SETTINGS,
                       lambda prompt=prompt: ollama_answer(model, prompt, BASE_SETTINGS))
        api_post("/api/generate", {"model": model, "keep_alive": 0})
    
    for model, size in OLLAMA_MODELS.items():
        run_ollama(model, size)


### Show a second interaction method: command line
Run the following in a terminal and screenshot the answer. It uses the same first prompt. Interactive CLI defaults can differ from the benchmark; this demonstrates the interface and is not a controlled timing run.

```bash
ollama run llama3.2:1b "A synthetic account has 20 failed SSH logins followed by one success from the same unknown IP. Give three defensive investigation steps. Separate facts from assumptions."
```

The previous cells demonstrate Python making direct API requests. Capture the command and output, not just the answer text.

### Hugging Face Transformers (Tasks 1–2, 5)
SmolLM2 is deliberately tiny and can give poor answers; evaluate them rather than assuming they are correct. Download the snapshot first, then measure loading from local files. This is a second hosting engine, not merely a different interface to Ollama.

In [ ]:
if ACTIVE_ENVIRONMENT == "jetstream2":
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
    from huggingface_hub import snapshot_download
    if DEVICE == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA is unavailable. Enable a GPU runtime or record the limitation and use DEVICE='cpu'.")
    print("Torch:", torch.__version__, "CUDA available:", torch.cuda.is_available())
    HF_PATH = snapshot_download(HF_MODEL, allow_patterns=["*.json", "*.safetensors", "*.txt", "*.model"],
                                local_files_only=OFFLINE_MODE)
    print("Downloaded snapshot:", HF_PATH)


In [ ]:
if ACTIVE_ENVIRONMENT == "jetstream2":
    # Release a previous model before re-running the load measurement.
    import gc
    if "hf_model" in globals():
        del hf_model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    
    def load_hf():
        global tokenizer, hf_model
        tokenizer = AutoTokenizer.from_pretrained(HF_PATH, local_files_only=True)
        hf_model = AutoModelForCausalLM.from_pretrained(HF_PATH, local_files_only=True).to(DEVICE)
        hf_model.eval()
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        return {"snapshot": str(HF_PATH), "dtype": str(next(hf_model.parameters()).dtype)}
    
    hf_load = record("transformers", HF_MODEL, 0.135, "", "load", BASE_SETTINGS, load_hf)
    
    def hf_answer(prompt, settings):
        set_seed(settings["seed"])
        text = tokenizer.apply_chat_template([{"role": "user", "content": prompt}],
                                             tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt").to(DEVICE)
        with torch.inference_mode():
            output = hf_model.generate(**inputs, max_new_tokens=settings["max_tokens"],
                                       do_sample=True, temperature=settings["temperature"],
                                       top_p=settings["top_p"], pad_token_id=tokenizer.eos_token_id)
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        tokens = output[0, inputs["input_ids"].shape[1]:]
        return {"answer": tokenizer.decode(tokens, skip_special_tokens=True),
                "output_tokens": len(tokens), "rate_definition": "output tokens / full client generation time"}
    
    if hf_load["status"] == "ok":
        for pid, prompt in PROMPTS.items():
            record("transformers", HF_MODEL, 0.135, pid, "baseline", BASE_SETTINGS,
                   lambda prompt=prompt: hf_answer(prompt, BASE_SETTINGS))


### Graphical interface (Tasks 1 and 3)
Run the next cell, use a planned prompt, and capture the prompt, model label, and answer.
The GUI uses Transformers/SmolLM2 and is separate from benchmark measurements.
Pause here for screenshots; this section's cleanup closes the interface.

Keep `ssh -N jetstream-lab-tunnel` open locally, then open `http://127.0.0.1:7860` on your workstation. Leave sharing disabled.


In [ ]:
if ACTIVE_ENVIRONMENT == "jetstream2":
    import gradio as gr
    COLAB_PUBLIC_LINK = False
    if "demo" in globals():
        demo.close()
    if "hf_model" not in globals():
        raise RuntimeError("Complete the Transformers loading cell before starting the GUI.")
    demo = gr.Interface(
        fn=lambda prompt: hf_answer(prompt, BASE_SETTINGS)["answer"],
        inputs=gr.Textbox(label="Your synthetic test prompt", value=PROMPTS["p1_security"], lines=4),
        outputs=gr.Textbox(label="Model answer", lines=8),
        title=f"IT7075 • {HF_MODEL}",
        description="Hugging Face Transformers hosting with a Gradio graphical interface.")
    demo.launch(server_name="127.0.0.1", server_port=7860, share=COLAB_PUBLIC_LINK,
                inline=True, prevent_thread_lock=True)


### Controlled settings experiment (Task 4)
Use the same model, prompt, and seed. Compare temperature **0.1 vs 1.2** while holding the other settings fixed; then compare **top-p 0.9 vs 0.3** at temperature 1.2. Similar answers are a legitimate outcome. Repeat with several seeds if you want stronger evidence; a single sample does not establish a trend.

In [ ]:
if ACTIVE_ENVIRONMENT == "jetstream2":
    SETTINGS_MODEL = "llama3.2:1b"
    variants = [dict(BASE_SETTINGS, temperature=0.1),
                dict(BASE_SETTINGS, temperature=1.2),
                dict(BASE_SETTINGS, temperature=1.2, top_p=0.3)]
    # Load outside response timing to keep these runs warm.
    api_post("/api/generate", {"model": SETTINGS_MODEL, "stream": False, "keep_alive": "10m",
                             "options": ollama_options(BASE_SETTINGS)})
    for settings in variants:
        record("ollama", SETTINGS_MODEL, 1.0, "p1_security", "settings", settings,
               lambda settings=settings: ollama_answer(SETTINGS_MODEL, PROMPTS["p1_security"], settings))
    api_post("/api/generate", {"model": SETTINGS_MODEL, "keep_alive": 0})


**My observations for this environment:** _Describe changes in wording, correctness, and format from the settings experiment. Quote actual responses._

### Larger-model experiment
Set `TRY_LARGER_MODEL=True` below for your Jetstream larger-model attempt. The model is
already downloaded. Use the same five prompts and record failures honestly. This model's
inference has not been setup-tested; the switch stays false until you choose to run it.


In [ ]:
if ACTIVE_ENVIRONMENT == "jetstream2":
    TRY_LARGER_MODEL = False
    if TRY_LARGER_MODEL:
        assert ENVIRONMENT == "jetstream2", "This optional experiment is intended for your class VM."
        large_model = "qwen2.5:7b"
        try:
            ensure_ollama_model(large_model)
            run_ollama(large_model, 7.0)
        except Exception as e:
            def failed_attempt():
                raise RuntimeError(str(e))
            record("ollama", large_model, 7.0, "", "setup", BASE_SETTINGS, failed_attempt)


### Export this environment and compare matching runs
Each environment has a separate result folder. Download its ZIP, then extract all environment folders into one `llmhosting_results` directory on your local machine. Rerun the comparison cell to combine them. Avoid duplicating folders; every execution is retained, including repeated trials.

The chart averages successful warm baseline responses across your prompts. Compare matching model/backend/device configurations and inspect variability; different output lengths can change total latency. Missing environments and failures are not zero-valued measurements. Do not use these averages as a quality score.
Only rows matching both the current prompt plan and the exact current baseline settings enter the chart. Other settings remain in `all_runs.csv`. Before comparing, retrieve Jetstream results with the command in `LLMHosting_Setup.md`; extract Colab and Jetstream ZIPs into the same `llmhosting_results` folder. Imported runs without recorded actual placement are marked `unverified` and summarized separately. Do not import setup-verification runs as your assignment evidence.


In [ ]:
if ACTIVE_ENVIRONMENT == "jetstream2":
    files = list(RESULTS_ROOT.glob("*/runs.jsonl"))
    rows = [json.loads(line) for file in files for line in file.read_text().splitlines() if line.strip()]
    df = pd.DataFrame(rows)
    if df.empty:
        print("No experiments recorded yet.")
    else:
        df.to_csv(OUT / "all_runs.csv", index=False)
        display(df[["environment", "device", "model", "prompt_id", "phase", "load_s", "response_s",
                    "tokens_s", "ram_peak_gib", "status"]])
        def canonical_settings(value):
            try:
                return json.dumps(json.loads(value) if isinstance(value, str) else value, sort_keys=True)
            except (TypeError, ValueError):
                return None
        matching_settings = df["settings"].map(canonical_settings) == json.dumps(BASE_SETTINGS, sort_keys=True)
        if "actual_device" not in df:
            df["actual_device"] = "unverified"
        df["actual_device"] = df["actual_device"].fillna("unverified")
        baseline = df[(df.status == "ok") & (df.phase == "baseline") &
                      (df.plan_id == PLAN_ID) & matching_settings]
        print("Successful baseline rows excluded for different settings:",
              int(((df.status == "ok") & (df.phase == "baseline") &
                   (df.plan_id == PLAN_ID) & ~matching_settings).sum()))
        if not baseline.empty:
            summary = baseline.groupby(["environment", "device", "actual_device", "backend", "model"]).agg(
                responses=("response_s", "count"), mean_response_s=("response_s", "mean"),
                std_response_s=("response_s", "std"), mean_tokens_s=("tokens_s", "mean"),
                max_system_ram_gib=("ram_peak_gib", "max"))
            display(summary)
            summary.to_csv(OUT / "comparison.csv")
            import matplotlib.pyplot as plt
            chart_values = summary["mean_response_s"].copy()
            chart_values.index = [f"{model.split('/')[-1]} ({backend})\n{env}: requested {requested}, actual {actual}"
                                  for env, requested, actual, backend, model in summary.index]
            ax = chart_values.plot.barh(figsize=(11, max(3.5, len(summary)*0.9)),
                        title="Warm response time — matching baseline runs")
            ax.set_xlabel("Mean client wall time (seconds); lower is faster")
            ax.set_ylabel("")
            plt.tight_layout()
            plt.savefig(OUT / "response_times.png", dpi=180, bbox_inches="tight")
            plt.show()
            plt.close(ax.figure)
        else:
            print("No successful baselines match the current prompt plan and settings.")
        failures = df[df.status == "error"]
        display(failures[["environment", "model", "phase", "error"]])


In [ ]:
if ACTIVE_ENVIRONMENT == "jetstream2":
    archive = shutil.make_archive(str(RESULTS_ROOT / f"llmhosting-{ENVIRONMENT}"), "zip",
                                  root_dir=OUT.parent, base_dir=ENVIRONMENT)
    print("Saved:", archive)
    # In Colab, download before disconnecting:
    if ENVIRONMENT == "colab":
        from google.colab import files
        files.download(archive)


### Finish this environment
Retrieve actual results with the command in `LLMHosting_Setup.md`. Stop the VM through the class dashboard when finished; closing the notebook or tunnel does not stop allocation usage.


In [ ]:
if ACTIVE_ENVIRONMENT == "jetstream2":
    if "demo" in globals():
        demo.close()
    print("Keep your exported results, screenshots, and notes for the report.")


### Final cross-environment analysis and submission checklist
Fill this table using your observations. If Jetstream2 is waived, state that explicitly rather than inventing results.

| Dimension | Local | Colab | Jetstream2 |
|---|---|---|---|
| Hardware and GPU placement | | | |
| Setup steps, time, and difficulty | | | |
| Speed (cite CSV rows/chart) | | | |
| Successful model sizes | | | |
| Failures and attempted fixes | | | |
| Cost / class allocation usage | | | |
| Privacy and where inference/data reside | | | |
| Reliability / session persistence | | | |

### Recommendations supported by your measurements
1. **Everyday security analyst:** _Choose an environment, cite measured results, and explain quality/speed tradeoffs._
2. **Sensitive or confidential data:** _Choose an environment and explain access controls and data location. Use synthetic data for this assignment._
3. **Quick experiment due today:** _Choose an environment and cite actual setup time, speed, and reliability._

### Troubleshooting journal
| Environment/task | Exact error or symptom | What I changed | Result / what I learned |
|---|---|---|---|
| | | | |

### Evidence checklist
- [ ] Plan and five unchanged prompts, including security and expected-failure prompts.
- [ ] Two hosting engines, with a graphical interface shown working.
- [ ] Three models, two or more families and sizes, one downloaded directly from Hugging Face.
- [ ] Five answers per model; outputs in every required environment.
- [ ] Two interaction methods, showing commands/prompts and answers.
- [ ] Low/high temperature and one other setting, with actual responses and interpretation.
- [ ] Hardware, load time, response time, throughput definition, RAM measurement method, and failures.
- [ ] Tables plus at least one chart; the six required dimensions: setup, speed, model sizes, cost, privacy, and reliability.
- [ ] Three recommendations backed by your own results.
- [ ] Screenshots labelled by task number, without passwords or personal data.
- [ ] One PDF report with video link on first page; notebook/ZIP are working files, not separate submissions.

**Suggested live video outline (5–10 minutes):** 0–1 min plan and models; 1–3 min local generation and GUI; 3–5 min Colab generation; 5–7 min Jetstream2 generation/larger-model attempt (if required); final 1–2 min chart, surprising result, and what you would change. Use live screens rather than slides of screenshots.

**Cleanup:** Close Gradio using the cleanup cell above. Download Colab files. Unload Ollama models if needed (`ollama stop MODEL`). Stop the Jetstream2 VM in the cloud dashboard. Keep your results and screenshots until grading is complete.

### Common fixes
- **Connection refused on port 11434:** start `ollama serve`, then retry the health check. If it says the port is already in use, check `ollama list`; another server may already be running.
- **Wrong kernel / missing imports:** select `.venv-llmhosting/bin/python`; installing into another Python environment will not fix this kernel.
- **Ollama reports CPU on Jetstream despite a GPU:** the prepared VM needed the Ollama service account in the existing `vglusers` device-access group, followed by a service restart. That fix is already applied. Inspect `ollama ps` and recorded placement; do not assume all VM images use the same group.
- **CUDA unavailable:** verify GPU allocation and PyTorch's CUDA build. Record the issue; do not label a CPU run as GPU.
- **Out of memory:** unload other Ollama models, close the Gradio interface, restart the kernel to release Transformers memory, and retry a smaller model. Record the original failure and the change.
- **Truncated answer:** 128 new tokens is the baseline limit. Keep it fixed for baseline comparisons; test a larger limit as a separately labelled settings experiment.
- **Gradio port already in use:** close the previous interface or change `server_port`; use the matching SSH tunnel port on a VM.
